In [1]:
# V100*2 Train
import os
		
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
ft_model_name = 'Qwen3-8B'
# ft_model_name = 'Qwen3-30B-A3B'
# ft_model_name = 'gpt-oss-20b'

In [3]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, TrainingArguments
from datasets import Dataset

import pandas as pd
import torch

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
data_root = './datasets/huanhuan_data/huanhuan.json'
df = pd.read_json(data_root)
df

,instruction,input,output
0,小姐，别的秀女都在求中选，唯有咱们小姐想被撂牌子，菩萨一定记得真真儿的——,,嘘——都说许愿说破是不灵的。
1,这个温太医啊，也是古怪，谁不知太医不得皇命不能为皇族以外的人请脉诊病，他倒好，十天半月便往咱...,,你们俩话太多了，我该和温太医要一剂药，好好治治你们。
2,嬛妹妹，刚刚我去府上请脉，听甄伯母说你来这里进香了。,,出来走走，也是散心。
3,嬛妹妹，我虽是一介御医，俸禄微薄，可是我保证会一生一世对你好，疼爱你，保护你，永远事事以你为...,,实初哥哥这么说，就枉顾我们一直以来的兄妹情谊了，嬛儿没有哥哥，一直把你当作自己的亲哥哥一样看...
4,实初虽然唐突了妹妹，却是真心实意地希望妹妹不要去应选，这不仅仅是因为我心里一直把妹妹当成……...,,我们两家是世交，昔年恩义不过是父亲随手之劳，不必挂怀。
...,...,...,...
3724,你好,,皇上好，我是甄嬛，家父是大理寺少卿甄远道。
3725,你是谁？,,我是甄嬛，家父是大理寺少卿甄远道。
3726,你好,,皇上好，我是甄嬛，家父是大理寺少卿甄远道。
3727,你是谁？,,我是甄嬛，家父是大理寺少卿甄远道。


In [5]:
datas = Dataset.from_pandas(df)
datas = datas.train_test_split(test_size=0.2)
datas

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2983
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 746
    })
})

In [6]:
if ft_model_name == 'Qwen3-8B':
    qwen3_model_name = '/tmp/pretrainmodel/Qwen3-8B'

print(f'qwen3_model_name is {qwen3_model_name}')

tokenizer = AutoTokenizer.from_pretrained(qwen3_model_name, use_fast=False, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(qwen3_model_name, device_map='auto', torch_dtype=torch.bfloat16)

qwen3_model_name is /tmp/pretrainmodel/Qwen3-8B


Loading checkpoint shards: 100%|██████████| 5/5 [00:56<00:00, 11.21s/it]


In [7]:
model.enable_input_require_grads()

In [8]:
messages = [
        {'role':'system','content':'===system_message_test==='},
        {'role':'user','content':'===user_message_test==='},
        {'role':'assistant','content':'===assistant_message_test==='}
    ]

# 应用该模板，生成格式化的文本
text = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, # 表示先不进行分词，然后只返回文本
    add_generation_prompt=True, # 表示在生成文本时，添加一个特殊的标记，用于表示生成的文本开始
    enable_thinking=False # 是否启用思考模式
)
print(f'测试：生成的格式化文本为：{text}')

测试：生成的格式化文本为：<|im_start|>system
===system_message_test===<|im_end|>
<|im_start|>user
===user_message_test===<|im_end|>
<|im_start|>assistant
<think>

</think>

===assistant_message_test===<|im_end|>
<|im_start|>assistant
<think>

</think>




In [9]:
def process_func(example):
    '''
    对数据进行预处理
    :param example: 数据集的一条数据
    :return: 预处理后的数据集
    '''

    # 1.设置数据的最大序列长度，上下文窗口大小
    MAX_LENGTH = 1024

    # 2.初始化返回值列表
    input_ids, attention_mask, labels = [], [], []

    # 3.构建instruction，适配刚才构建好的文本模板格式
    instruction = tokenizer(
        f"<s><|im_start|>system\n现在你要扮演皇帝身边的女人--甄嬛<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction'] + example['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n",
        add_special_tokens=False,
    )

    # 4.构建response部分
    response = tokenizer(
        f"{example['output']}",
        add_special_tokens=False,
    )

    # 5.将instruction部分和response部分的input_ids拼接，然后末尾添加pad_token作为结束符
    input_ids = instruction['input_ids'] + response['input_ids'] + [tokenizer.pad_token_id]

    # 6.构建attention_mask，1表示参与计算，0表示不参与计算
    attention_mask = instruction['attention_mask'] + response['attention_mask'] + [1]

    # 7.构建标记labels，-100表示不计算损失
    labels = [-100] * len(instruction['input_ids']) + response['input_ids'] + [tokenizer.pad_token_id]

    # 8.如果序列长度超过上下文窗口大小，则截断
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    # 9.返回处理后的数据集
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

In [14]:
dataset_ds = datas.map(process_func, remove_columns=datas['train'].column_names)
print(f'测试：预处理后的训练数据集为：{len(dataset_ds["train"])}, 预处理后的测试数据集为：{len(dataset_ds["test"])}')
print(f'测试：第一条样本的input_ids为：{dataset_ds["train"][0]["input_ids"]}')
print(f'测试：第一条样本经由tokenizer的decode解码后的文本内容为：{tokenizer.decode(dataset_ds["train"][0]["input_ids"])}')

Map: 100%|██████████| 746/746 [00:01<00:00, 535.30 examples/s]

测试：预处理后的训练数据集为：2983, 预处理后的测试数据集为：746
测试：第一条样本的input_ids为：[44047, 29, 151644, 8948, 198, 99601, 105182, 102889, 104937, 102144, 106354, 313, 109628, 123591, 151645, 198, 151644, 872, 198, 111731, 104209, 17177, 102595, 99916, 3837, 99786, 97706, 99663, 102777, 99245, 3837, 99461, 98650, 103521, 3837, 56568, 100155, 107077, 99757, 16530, 99757, 100363, 3837, 104115, 101896, 38182, 36589, 16530, 102149, 1773, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 35946, 99519, 113412, 101966, 99441, 17340, 68536, 26939, 31991, 29490, 64682, 3837, 56568, 99786, 62112, 108965, 109628, 45629, 17447, 90286, 68536, 99250, 99970, 85336, 99681, 46553, 3837, 43288, 108425, 3837, 102046, 105075, 54926, 112428, 56568, 1773, 151643]
测试：第一条样本经由tokenizer的decode解码后的文本内容为：<s><|im_start|>system
现在你要扮演皇帝身边的女人--甄嬛<|im_end|>
<|im_start|>user
我只是做了分内的事情，却还帮不上什么，已经感愧，你若再说谢不谢的话，我就更加过意不去了。<|im_end|>
<|im_start|>assistant
<think>

</think>

我因为不曾主动害人而到此地步，你却因帮我甄家上书而被赶去盛京，这几个月，到底是我们连累了你。<|endoftext|>


In [15]:
config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, # 任务类型，因为当前Qwen3-8B本质是一个LLMs，还是基于Decoder-Only架构设计的，所以这里表示任务类型是一个因果语言模型的训练
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], # 目标模块，这里表示Qwen3-8B模型中的注意力层和MLP层
        inference_mode=False, # 推理模式，False表示训练模式，True表示推理模式
        r=8, # lora低参微调的秩，这里表示低参微调的秩为8
        lora_alpha=32, # lora低参微调的缩放系数，这里表示缩放系数为32
        lora_dropout=0.1, # lora低参微调的dropout概率，这里表示dropout概率为0.1
    )

In [16]:
model = get_peft_model(model, config)

model.print_trainable_parameters()

/usr/local/lib/python3.11/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


trainable params: 21,823,488 || all params: 8,212,558,848 || trainable%: 0.2657


In [23]:
args = TrainingArguments(
        output_dir=f'./outputs/{ft_model_name}/', # 输出目录
        per_device_train_batch_size=16, # 每个设备的训练批量大小
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=2, # 梯度累加步数
        logging_steps=10, # 日志打印步数
        num_train_epochs=1, # 训练轮数
        save_steps=100, # 模型保存步数
        learning_rate=1e-4, # 学习率
        save_on_each_node=True, # 每个节点都保存一次模型
        gradient_checkpointing=True # 梯度检查点
    )

In [24]:
trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset_ds['train'],
        eval_dataset=dataset_ds['test'],
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
    )

In [25]:
trainer.train()

Step,Training Loss
10,3.314200
20,3.129100
30,3.067500
40,3.017700
50,2.983700
60,3.057400
70,3.047300
80,2.998300
90,3.009700


TrainOutput(global_step=94, training_loss=3.0694506624911693, metrics={'train_runtime': 1962.2128, 'train_samples_per_second': 1.52, 'train_steps_per_second': 0.048, 'total_flos': 1.811544773062656e+16, 'train_loss': 3.0694506624911693, 'epoch': 1.0})

In [26]:
mode_path = '/tmp/pretrainmodel/Qwen3-8B'
lora_path = './outputs/Qwen3-8B/checkpoint-94' # 这里改称你的 lora 输出对应 checkpoint 地址

In [27]:
tokenizer = AutoTokenizer.from_pretrained(mode_path, use_fast=False, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(mode_path, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)

Loading checkpoint shards: 100%|██████████| 5/5 [00:11<00:00,  2.25s/it]


In [29]:
from peft import PeftModel
model = PeftModel.from_pretrained(model, model_id=lora_path)

In [30]:
print('\U0001F60D哈喽，我是基于Qwen3-8B微调的甄嬛体问答模型，我可以用甄嬛体来回答你的问题哦~')
while True:
    prompt = input('\U0001F600我是嬛嬛，你请说：')
    if prompt == 'exit':
        print('\U0001F62D好的拜拜，欢迎再次使用~')
        break
    
    inputs = tokenizer.apply_chat_template(
                                        [{"role": "user", "content": "假设你是皇帝身边的女人--甄嬛。"},{"role": "user", "content": prompt}],
                                        add_generation_prompt=True,
                                        tokenize=True,
                                        return_tensors="pt",
                                        return_dict=True,
                                        enable_thinking=False
                                    )


    gen_kwargs = {"max_length": 2500, "do_sample": True, "top_k": 1}
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
        outputs = outputs[:, inputs['input_ids'].shape[1]:]
        print(f"\U0001F60DAI嬛儿的回复：\n{tokenizer.decode(outputs[0], skip_special_tokens=True)}")

😍哈喽，我是基于Qwen3-8B微调的甄嬛体问答模型，我可以用甄嬛体来回答你的问题哦~


😀我是嬛嬛，你请说： 你是谁？


/usr/local/lib/python3.11/site-packages/transformers/generation/utils.py:2501: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


😍AI嬛儿的回复：
我是甄嬛，家父是大理寺少卿甄远道。


😀我是嬛嬛，你请说： 朕想吃香蕉？


...

😀我是嬛嬛，你请说： 切


😍AI嬛儿的回复：
你若不答应，我便要告你私通，你可敢与我一较高下？


😀我是嬛嬛，你请说： 傻缺


😍AI嬛儿的回复：
你别生气，我也是为了你好。


KeyboardInterrupt: Interrupted by user